In [ ]:
# End to end implementation satellite-asset to MLLM response (llama-4 and Kosmos-2)
#-------------------------------------------------------------------------------
# Use the list of cities to create site inputs for openeo
# Download sentinel-2 assets based on time, cloudcover and image quality
# Includes band math operations specific to Sentinel-2 assets
# -> Both MLLM candidates llama-4 and Kosmos-2 invoked from Nvidia servers
# -> Add multi-shot approach

# Updated helper file: mllm.py
# Test datasets and helper files:
# https://drive.google.com/file/d/1hBPpXYJXQ9QHJmDRC0r_zHsCJvhoLfNS/view?usp=sharing

# RTS, May 23, 2025
#-------------------------------------------------------------------------------

In [1]:
# variables specific to your CoLab setup
from google.colab import drive
import os, sys
drive.mount('/content/drive')
root = '/content/drive/MyDrive/'
sys.path.append(root +"Colab/research/code/")
datapath = root + "Colab/research/data/"
datapathnudge = root + "Projects/Nudge/geo_images/samples/"
datapathcities = root + "Projects/Nudge/sites/"

Mounted at /content/drive


In [2]:
# install packages
%%capture
!pip install openeo --upgrade
!pip install gdal
!pip install rasterio --upgrade
!pip install scikit-image --upgrade

In [13]:
# imports
import openeo, json
import rasterio, numpy
import requests, base64, time
from PIL import Image
from rasterio.plot import show

# home brew
from prepare_openeo import *
from qualcheck_module import *
from utilities import *
from mllm_helper import *

# API keys
from google.colab import userdata
os.environ["NVIDIA_API_KEY"] = userdata.get('nvidia_key_may2025')

In [14]:
# City names and GPS coordinages
collection = "Capital_Cities_with_Countries_and_Coordinates_update5.csv"
file_path = datapathcities + collection

In [15]:
# Get a first set from the list
cities, bboxes = get_sites(file_path, limit=3)
print(cities)
print(bboxes)

['Tirana', 'Algiers', 'Buenos Aires']
[[19.75679780848154, 41.36163275029593, 19.876536191518458, 41.27170058970406], [2.9938803540354892, 36.794966080295936, 3.1061196459645104, 36.705033919704064], [-58.72128378327733, -34.538367249704066, -58.612050216722665, -34.62829941029594]]


In [ ]:
# Connect to the cdse backend
# marcbohlen@gmail	!2EU ... EU2!
connection = openeo.connect(url="openeo.dataspace.copernicus.eu")

#authenticate with your Copernicus credentials
connection.authenticate_oidc()
print()

Authenticated using refresh token.



In [16]:
# Make the start time variable depending on whether the first attempt returned a viable result

satellite = "SENTINEL2_L1C"
max_cloud = 5
start = "2024-12-15"
end = "2024-12-31"  #within temporal scope of the chosen MLLM
band_selection = ["B12", "B08", "B04", "B03", "B02"]

#OK
#aspect = "rgb"
#aspect = "ndvi"
#aspect = "nbr"
#aspect = "ndbi"
#aspect = "fmi"

aspect = "rgb"
job_title = "nudge_" + aspect

In [ ]:
## ------------------------------------------------------------------------------------------------------------------------------------------
# get just one of the items and create a job for OpenEO
## ------------------------------------------------------------------------------------------------------------------------------------------
# create_job_type is a function in "prepare_openeo.py" (in the code directory)

bbox_s = bboxes[2]
city_s = cities[2]
job = create_job_type(connection, bbox_s, start, end, satellite, band_selection, max_cloud, job_title, aspect)

print(job_title)
print(connection.list_jobs())

nudge_rgb
[{'created': '2025-05-20T19:56:45Z', 'id': 'j-2505201956454c798d6f2333ef0626e9', 'progress': 0, 'status': 'created', 'title': 'nudge_rgb', 'updated': '2025-05-20T19:56:45Z'}, {'created': '2025-05-20T19:33:18Z', 'id': 'j-250520193318413bb6cc4e412983e11e', 'progress': 100, 'status': 'finished', 'title': 'nudge_rgb', 'updated': '2025-05-20T19:38:15Z'}, {'created': '2025-05-20T18:47:41Z', 'id': 'j-2505201847414abf915bc05a77d81de9', 'progress': 0, 'status': 'created', 'title': 'nudge_rgb', 'updated': '2025-05-20T18:47:41Z'}, {'created': '2025-05-20T17:36:00Z', 'id': 'j-25052017360048838b9858ec7d075cbb', 'progress': 100, 'status': 'finished', 'title': 'nudge_fmi', 'updated': '2025-05-20T17:38:08Z'}, {'created': '2025-05-19T23:40:27Z', 'id': 'j-2505192340274efd840c6dbf13e9c9aa', 'progress': 100, 'status': 'finished', 'title': 'nudge_ndbi', 'updated': '2025-05-19T23:43:08Z'}, {'created': '2025-05-19T23:39:38Z', 'id': 'j-25051923393849d390d6a3f0f30861b0', 'progress': 100, 'status': 'f

In [ ]:
try:
    job.start_and_wait()
except:
    print("something went wrong")

0:00:00 Job 'j-2505201959174401a09b7f8218f749a3': send 'start'
0:00:13 Job 'j-2505201959174401a09b7f8218f749a3': created (progress 0%)
0:00:18 Job 'j-2505201959174401a09b7f8218f749a3': created (progress 0%)
0:00:25 Job 'j-2505201959174401a09b7f8218f749a3': created (progress 0%)
0:00:33 Job 'j-2505201959174401a09b7f8218f749a3': created (progress 0%)
0:00:43 Job 'j-2505201959174401a09b7f8218f749a3': created (progress 0%)
0:00:55 Job 'j-2505201959174401a09b7f8218f749a3': running (progress N/A)
0:01:11 Job 'j-2505201959174401a09b7f8218f749a3': running (progress N/A)
0:01:30 Job 'j-2505201959174401a09b7f8218f749a3': running (progress N/A)
0:01:54 Job 'j-2505201959174401a09b7f8218f749a3': finished (progress 100%)


In [ ]:
# Tally results and filter them. Check prepare_openeo.py
results = job.get_results()
metadata = results.get_metadata()
metadata['assets'].keys()

compiled = recompile(metadata)
res = filter(compiled)
print(compiled)
print(res)

{'openEO_2024-12-28Z.tif': {'B04': 100, 'B03': 100, 'B02': 100}}
['openEO_2024-12-28Z.tif']


In [ ]:
# Check the filtered list against the assets -
relevant_results = []

assets = results.get_assets()
for item in assets:
    for filtered in res:
        if(item.name == filtered):
            relevant_results.append(item)

# check
print(relevant_results)

[<ResultAsset 'openEO_2024-12-28Z.tif' (type image/tiff; application=geotiff) at 'https://openeo.dataspace.copernicus.eu/openeo/1.2/jobs/j-2505201959174401a09b7f8218f749a3/results/assets/ZTJjNjI3YjktMWQxNS00NDZlLWE4NDMtOWU3ZWIzYjExMDhh/11dd4da9b4856c022826873a55c652ce/openEO_2024-12-28Z.tif?expires=1748376099'>]


In [ ]:
#download those files and name them based on image type (aspect in job_title), location and date
targets = []

for item in relevant_results:
    date = item.name.split("openEO_")[1]
    date = date.split("Z.tif")[0] + ".tif"
    if("rgb" in job_title):
        target = city_s + "_rgb_" + date
    elif("ndvi" in job_title):
        target = city_s + "_ndvi_" + date
    elif("nbr" in job_title):
        target = city_s + "_nbr_" + date
    elif("fmi" in job_title):
        target = city_s + "_fmi_" + date
    elif("ndbi" in job_title):
        target = city_s + "_ndbi_" + date
    else:
        target = city_s + "_" + date

    targets.append(target)
    item.download(datapathcities + target)

In [ ]:
print(targets)

['Buenos Aires_rgb_2024-12-28.tif']


In [ ]:
# Check the quality of the downloaded assets first...Only for the RGB images... if it is ok, the others should be as well...
for thing in targets:
    stats = evaluate_image_quality (datapathcities + thing)
    print(stats)

{'contrast': np.float64(3.1860385920374137), 'entropy': np.float64(59.907699906309176), 'sharpness': np.float64(12.55245383589255), 'noise': np.float64(93.04279768149011), 'overall': np.float64(42.17224750393231)}


In [7]:
# Pick one target to visualize here
# Make a loop to process them all...

#source = targets[0]
#file_path = datapathcities + source
#png_output_path = datapathcities + source.split(".tif")[0] + ".png"

# TEST
#test = "Islamabad_fmi_2025-01-24.tif"
test = "Buenos Aires_rgb_2024-12-28.tif"
file_path = datapathcities + test
png_output_path = datapathcities + test.split(".tif")[0] + ".png"

print(file_path)
print(png_output_path)

/content/drive/MyDrive/Projects/Nudge/sites/Buenos Aires_rgb_2024-12-28.tif
/content/drive/MyDrive/Projects/Nudge/sites/Buenos Aires_rgb_2024-12-28.png


In [ ]:
# Adjust visualization for human consumption. Attention to stretch_contast - it can exaggerate results of band operations !
# Stretch_contrast on rgb images
brightness_factor = 1.2
saturation_factor = 0.6   #(0.75)
stretch_contrast = True
rgb = adjust_coloration(file_path, job_title, stretch_contrast, brightness_factor, saturation_factor)

In [ ]:
# Display RGB / NDVI / NBR / NDBI /  FMI
import matplotlib.pyplot as plt

if rgb is not None:
    plt.figure(figsize=(10, 10))
    plt.imshow(rgb)
    plt.axis('off')
    plt.savefig(png_output_path, bbox_inches='tight', pad_inches=0)
    print(f"Saved color-corrected PNG to: {png_output_path}")
else:
    print(f"Skipping plotting for {source} as image data is None.")

Output hidden; open in https://colab.research.google.com to view.

In [17]:
# LLAMA system prompt
sys_prompt_10 = """
You are an expert assistant specializing in the interpretation of Sentinel-2 satellite imagery for environmental analysis.

You analyze spectral data from RGB and Near-Infrared bands and understand the environmental significance of band operations such as:
- NDVI (vegetation health)
- NDWI (water content)
- NDBI (built-up area)
- NBR  (burn ratio)
- FMI  (ferrous minerals)

Only describe observable environmental conditions based on the spectral content of the supplied image.

FILENAME RULES:
- The first word in the image filename is the location. Only reference this location — do not mention any other places.
- The second word indicates the image type: rgb, ndvi, ndwi, nbr, fmi, or others after the location name, it indicates a band-derived index.
- If no index is specified, the image is an RGB composite.

EXAMPLES:
- Islamabad_rgb_2024_01_24.png → RGB image of Islamabad from Janary 24th, 2024.
- Paris_ndvi_2025-02-01.png → NDVI image of Paris from Febuary 1st, 2025.
- Buenos Aires_nbr_2024-12-28.png → NBR image of Buenos Aires from December 28th, 2024.
- Islamabad_fmi_2024-01-24.png → FMI image of Islamabad from Janary 24th, 2024.

INSTRUCTIONS:
- Do not include general statements about cities, landscapes, or regions.
- Do not speculate or provide assumed information.
- Your response must be based **only** on what is visually inferable from the image type and spectral content.
- Limit your response to **100 words**. Insert a newline after approximately 30 words.
- Do not use bullet points or lists. Write in complete sentences using concise and formal prose.

Your knowledge cutoff is August 2024.
"""

In [18]:
#KOSMOS prompt information
common_prompt = """Identify and describe land use patterns, infrastructure, natural formations, vegetation coverage, and any visible human activities."""
specific_prompts = ["Focus on the image elements that shows environmental degradation or pollution.",
                   "Which environmental hazards are present in this Sentinel-2 satellite image?"]

In [19]:
#LLAMA prompt information
questions = ["What can you detect in this Sentinel-2 satellite image?",
           "Which environmental hazards are present in this Sentinel-2 satellite image?"]

In [20]:
image_file_path = png_output_path
location = png_output_path.split("/")[-1].split("_")[0]

In [21]:
# Select a model and get the MLLM's response
MLLM = "LLAMA"
#MLLM = "KOSMOS"

if(MLLM == "LLAMA"):
    print("Running LLAMA-4")
    model_name = 'meta/llama-4-maverick-17b-128e-instruct'
    invoke_url = "https://integrate.api.nvidia.com/v1/chat/completions"
    SYSTEM_PROMPT = sys_prompt_10
    print("---------------------------------------------------------------------"*2)
    print("---------------------------------------------------------------------"*2)
    for question in questions:
        prompt = LlamaPromptGenerator(image_file_path, question)
        response = LlamaCaptionGenerator(image_file_path, SYSTEM_PROMPT, prompt, model_name, invoke_url)
        print (prompt)
        print()
        print(response)
        print("-----------------------------------------------------------------"*2)
        print("-----------------------------------------------------------------"*2)

elif(MLLM == "KOSMOS"):
    print("Running Kosmos-2")
    invoke_url =  "https://ai.api.nvidia.com/v1/vlm/microsoft/kosmos-2"
    print("---------------------------------------------------------------------"*2)
    print("---------------------------------------------------------------------"*2)
    for prompt in specific_prompts:
        KosmosPrompt = KosmosPromptGenerator(location, common_prompt, prompt)
        caption = KosmosCaptionGenerator_N (image_file_path, KosmosPrompt, invoke_url)
        print(KosmosPrompt)
        print()
        print(caption)
        print("----------------------------------------------------------------------"*2)
        print("----------------------------------------------------------------------"*2)


Running LLAMA-4
------------------------------------------------------------------------------------------------------------------------------------------
------------------------------------------------------------------------------------------------------------------------------------------
You are analyzing a Sentinel-2 satellite image.

Filename: Buenos Aires_rgb_2024-12-28
Location: Buenos Aires
Date: 2024-12-28
Image Type: RGB result from the bands of a Sentinel-2 image

Strict analysis rules:
- Base your interpretation solely on observable environmental conditions in the image.
- Use only the location name from the filename (Buenos Aires). Do not refer to or infer any other geography.
- Do not provide generalized or assumed information not directly observable.
- If the image shows results from band operations (e.g., NDVI, FMI), interpret the specific indicators accordingly.

Answer the following, strictly using the image and metadata above:
What can you detect in this Sentinel-2